# Tiempos totales programados y calibrados de las rutas QroBus

Este notebook calcula la duración completa de cada viaje usando `stop_times.txt`, resume P10, P50, promedio y P90 por ruta y sentido y reconstruye tiempos acumulados desde cada parada. Si existe el histórico local de Google, aprende una calibración robusta `constante + factor × GTFS` para producir escenarios P50 y P80. No realiza llamadas nuevas a Google Maps API.

> Como el GTFS local no contiene `shapes.txt`, el mapa toma el viaje más cercano a la mediana y une sus paradas consecutivas siguiendo la red vial de OpenStreetMap, con los trazos que `01_PromedioRutas.ipynb` deja en `data/processed/segmentos_red_vial.csv`. Es el recorrido más probable por calles, no el derrotero oficial; si ese archivo no existe, las paradas se unen con líneas rectas.


In [1]:
from pathlib import Path
import colorsys
import html
import sys

import folium
import numpy as np
import pandas as pd
from IPython.display import display

# Los módulos compartidos viven en src/scripts/; funciona al ejecutar desde
# src/notebooks/, desde src/ o desde la raíz del repositorio.
SCRIPTS_DIR = next(
    base / "scripts"
    for base in [Path.cwd().parent, Path.cwd(), Path.cwd() / "src"]
    if (base / "scripts" / "config.py").exists()
)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from config import (
    FACTOR_DESVIO_CAMINATA,
    GTFS_DIR,
    MAPS_DIR,
    PROCESSED_DIR,
    VELOCIDAD_CAMINATA_M_MIN,
)
from geo import cargar_trazos, haversine_m
from gtfs_network import gtfs_time_to_seconds

RESUMEN_PATH = PROCESSED_DIR / "tiempos_totales_rutas_gtfs.csv"
VIAJES_PATH = PROCESSED_DIR / "duraciones_viajes_gtfs.csv"
OBSERVACIONES_GOOGLE_PATH = PROCESSED_DIR / "google_maps_transit_observaciones.csv"
CALIBRACION_PATH = PROCESSED_DIR / "calibracion_tiempos_rutas_google.csv"
ACUMULADOS_PATH = PROCESSED_DIR / "tiempos_acumulados_paradas_rutas.csv"
MAPA_PATH = MAPS_DIR / "mapa_tiempos_totales_rutas_gtfs.html"
TRAZOS_PATH = PROCESSED_DIR / "segmentos_red_vial.csv"
MIN_OBSERVACIONES_RUTA = 30
MIN_RANGO_GTFS_RUTA_MIN = 10
SEMIVIDA_OBSERVACIONES_DIAS = 14
CONEXIONES_PATH = PROCESSED_DIR / "conexiones_paradas_rutas.csv"
# Una parada a 200 m o menos en línea recta (unos 250 m por calles, ~4 min a
# 4 km/h) se ofrece como conexión caminando. El cálculo de tiempos solo
# transborda en la misma parada. Velocidad y desvío vienen de config.py.
RADIO_CONEXION_A_PIE_M = 200
print(f"GTFS: {GTFS_DIR}")
print(f"Mapa: {MAPA_PATH}")


GTFS: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/gtfs
Mapa: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_tiempos_totales_rutas_gtfs.html


## Funciones del análisis

Las funciones quedan definidas dentro del notebook para que el análisis no dependa de un script externo.


In [2]:
def calcular_duraciones_viajes(stop_times, trips, routes):
    """Devuelve una fila por viaje con su duración programada completa."""
    stop_times = stop_times.copy()
    stop_times["arrival_sec"] = gtfs_time_to_seconds(stop_times["arrival_time"])
    stop_times["departure_sec"] = gtfs_time_to_seconds(stop_times["departure_time"])
    stop_times["stop_sequence"] = pd.to_numeric(
        stop_times["stop_sequence"], errors="raise"
    )
    ordenados = stop_times.sort_values(["trip_id", "stop_sequence"])
    extremos = ordenados.groupby("trip_id", as_index=False).agg(
        stop_id_inicio=("stop_id", "first"),
        stop_id_fin=("stop_id", "last"),
        salida_inicio_sec=("departure_sec", "first"),
        llegada_fin_sec=("arrival_sec", "last"),
        numero_paradas=("stop_id", "size"),
    )
    extremos["duracion_programada_min"] = (
        extremos["llegada_fin_sec"] - extremos["salida_inicio_sec"]
    ) / 60
    invalidos = extremos["duracion_programada_min"].lt(0)
    if invalidos.any():
        ejemplos = extremos.loc[invalidos, "trip_id"].head().tolist()
        raise ValueError(f"Hay viajes con duración negativa: {ejemplos}")
    viajes = (
        extremos.merge(trips, on="trip_id", how="left", validate="one_to_one")
        .merge(routes, on="route_id", how="left", validate="many_to_one")
    )
    if viajes["route_id"].isna().any():
        raise ValueError("Hay trip_id de stop_times que no existen en trips.txt.")
    return viajes


def resumir_por_ruta_sentido(viajes):
    """Resume la variación programada sin ocultarla en un único promedio."""
    def percentil(q):
        return lambda serie: float(np.nanpercentile(serie, q))

    agrupadores = ["route_id", "route_short_name", "route_long_name", "direction_id"]
    resumen = viajes.groupby(
        agrupadores, dropna=False, as_index=False
    ).agg(
        numero_viajes=("trip_id", "nunique"),
        numero_servicios=("service_id", "nunique"),
        duracion_min_min=("duracion_programada_min", "min"),
        duracion_p10_min=("duracion_programada_min", percentil(10)),
        duracion_mediana_min=("duracion_programada_min", "median"),
        duracion_promedio_min=("duracion_programada_min", "mean"),
        duracion_p90_min=("duracion_programada_min", percentil(90)),
        duracion_max_min=("duracion_programada_min", "max"),
        paradas_mediana=("numero_paradas", "median"),
    )
    columnas_tiempo = [c for c in resumen if c.startswith("duracion_")]
    resumen[columnas_tiempo] = resumen[columnas_tiempo].round(2)
    resumen["paradas_mediana"] = resumen["paradas_mediana"].round(1)
    resumen["estado_calculo"] = "calculado"
    return resumen.sort_values(["route_short_name", "direction_id"])


def agregar_rutas_sin_viajes(resumen, routes):
    """Conserva el catálogo completo e identifica rutas sin horarios."""
    route_ids_con_viajes = set(resumen["route_id"].dropna().astype(str))
    faltantes = routes.loc[
        ~routes["route_id"].astype(str).isin(route_ids_con_viajes),
        ["route_id", "route_short_name", "route_long_name"],
    ].copy()
    if faltantes.empty:
        return resumen
    faltantes["direction_id"] = np.nan
    faltantes["numero_viajes"] = 0
    faltantes["numero_servicios"] = 0
    for columna in [
        "duracion_min_min", "duracion_p10_min", "duracion_mediana_min",
        "duracion_promedio_min", "duracion_p90_min", "duracion_max_min",
        "paradas_mediana",
    ]:
        faltantes[columna] = np.nan
    faltantes["estado_calculo"] = "sin viajes en trips.txt"
    return (
        pd.concat([resumen, faltantes[resumen.columns]], ignore_index=True)
        .sort_values(["route_short_name", "direction_id"], na_position="last")
        .reset_index(drop=True)
    )


### Calibración robusta con el histórico de Google

El modelo restringe la constante a valores no negativos y el factor a valores mayores o iguales que uno, porque su objetivo es representar atraso adicional y nunca reducir el horario GTFS. P50 y P80 se ajustan por separado mediante pérdida cuantílica y las observaciones recientes reciben mayor peso.


In [3]:
def percentil_ponderado(valores, pesos, q):
    """Calcula un percentil con pesos no negativos."""
    valores = np.asarray(valores, dtype=float)
    pesos = np.asarray(pesos, dtype=float)
    validos = np.isfinite(valores) & np.isfinite(pesos) & (pesos > 0)
    if not validos.any():
        return np.nan
    valores, pesos = valores[validos], pesos[validos]
    orden = np.argsort(valores)
    valores, pesos = valores[orden], pesos[orden]
    acumulado = np.cumsum(pesos)
    posicion = np.searchsorted(acumulado, q * acumulado[-1], side="left")
    return float(valores[min(posicion, len(valores) - 1)])


def preparar_observaciones_google(observaciones):
    """Limpia el histórico y calcula pesos de recencia sin llamar a la API."""
    requeridas = {
        "observed_at_utc", "route_id_principal", "tiempo_gtfs_min",
        "google_eta_min", "condition", "status_code",
    }
    faltantes = requeridas - set(observaciones.columns)
    if faltantes:
        raise ValueError(f"El histórico de Google no contiene: {sorted(faltantes)}")
    datos = observaciones.copy()
    datos["route_id_principal"] = (
        datos["route_id_principal"].astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    datos["tiempo_gtfs_min"] = pd.to_numeric(datos["tiempo_gtfs_min"], errors="coerce")
    datos["google_eta_min"] = pd.to_numeric(datos["google_eta_min"], errors="coerce")
    datos["status_code"] = pd.to_numeric(datos["status_code"], errors="coerce")
    datos["observed_at_utc"] = pd.to_datetime(
        datos["observed_at_utc"], errors="coerce", utc=True, format="mixed"
    )
    datos = datos[
        datos["condition"].eq("ROUTE_EXISTS")
        & datos["status_code"].fillna(0).eq(0)
        & datos["tiempo_gtfs_min"].between(1, 180, inclusive="both")
        & datos["google_eta_min"].between(1, 180, inclusive="both")
        & datos["observed_at_utc"].notna()
    ].copy()
    razon = datos["google_eta_min"] / datos["tiempo_gtfs_min"]
    datos = datos[razon.between(0.25, 4, inclusive="both")].copy()
    if datos.empty:
        raise ValueError("No existen observaciones válidas para calibrar tiempos.")
    referencia = datos["observed_at_utc"].max()
    edad_dias = (referencia - datos["observed_at_utc"]).dt.total_seconds() / 86400
    datos["peso_recencia"] = 0.5 ** (edad_dias / SEMIVIDA_OBSERVACIONES_DIAS)
    return datos


def ajustar_modelo_cuantil(datos, q):
    """Ajusta Google = constante + factor × GTFS con restricciones de atraso."""
    x = datos["tiempo_gtfs_min"].to_numpy(dtype=float)
    y = datos["google_eta_min"].to_numpy(dtype=float)
    pesos = datos["peso_recencia"].to_numpy(dtype=float)
    limite_inferior, limite_superior = 1.0, 2.5
    mejor = None
    for _ in range(3):
        factores = np.linspace(limite_inferior, limite_superior, 301)
        for factor in factores:
            constante = max(0.0, percentil_ponderado(y - factor * x, pesos, q))
            error = y - (constante + factor * x)
            perdida = np.sum(
                pesos * np.where(error >= 0, q * error, (q - 1) * error)
            )
            if mejor is None or perdida < mejor[0]:
                mejor = (float(perdida), float(constante), float(factor))
        paso = (limite_superior - limite_inferior) / 300
        limite_inferior = max(1.0, mejor[2] - 2 * paso)
        limite_superior = min(2.5, mejor[2] + 2 * paso)
    return mejor[1], mejor[2]


def calibrar_modelos_por_ruta(observaciones):
    """Ajusta modelos por ruta y los contrae hacia un modelo global."""
    global_p50 = ajustar_modelo_cuantil(observaciones, 0.50)
    global_p80 = ajustar_modelo_cuantil(observaciones, 0.80)
    registros = [{
        "route_id": "__GLOBAL__",
        "observaciones_google": int(len(observaciones)),
        "rango_gtfs_observado_min": float(
            observaciones["tiempo_gtfs_min"].max()
            - observaciones["tiempo_gtfs_min"].min()
        ),
        "peso_modelo_ruta": 0.0,
        "constante_p50_min": global_p50[0],
        "factor_p50": global_p50[1],
        "constante_p80_min": max(global_p80[0], global_p50[0]),
        "factor_p80": max(global_p80[1], global_p50[1]),
        "fuente_modelo": "global",
    }]
    for route_id, grupo in observaciones.dropna(
        subset=["route_id_principal"]
    ).groupby("route_id_principal"):
        rango = float(grupo["tiempo_gtfs_min"].max() - grupo["tiempo_gtfs_min"].min())
        if len(grupo) < MIN_OBSERVACIONES_RUTA or rango < MIN_RANGO_GTFS_RUTA_MIN:
            continue
        ruta_p50 = ajustar_modelo_cuantil(grupo, 0.50)
        ruta_p80 = ajustar_modelo_cuantil(grupo, 0.80)
        peso = (len(grupo) / (len(grupo) + 100)) * min(1.0, rango / 20)
        constante_p50 = (1 - peso) * global_p50[0] + peso * ruta_p50[0]
        factor_p50 = (1 - peso) * global_p50[1] + peso * ruta_p50[1]
        constante_p80 = (1 - peso) * global_p80[0] + peso * ruta_p80[0]
        factor_p80 = (1 - peso) * global_p80[1] + peso * ruta_p80[1]
        registros.append({
            "route_id": str(route_id),
            "observaciones_google": int(len(grupo)),
            "rango_gtfs_observado_min": rango,
            "peso_modelo_ruta": float(peso),
            "constante_p50_min": float(constante_p50),
            "factor_p50": float(factor_p50),
            "constante_p80_min": float(max(constante_p80, constante_p50)),
            "factor_p80": float(max(factor_p80, factor_p50)),
            "fuente_modelo": "ruta suavizada con global",
        })
    return pd.DataFrame(registros)


In [4]:
def seleccionar_viajes_representativos(viajes, resumen):
    """Elige por ruta y sentido el viaje más cercano a la duración mediana."""
    calculados = resumen.loc[resumen["estado_calculo"].eq("calculado")].copy()
    calculados["direction_key"] = (
        pd.to_numeric(calculados["direction_id"], errors="coerce")
        .astype("Int64").astype("string").fillna("sin_sentido")
    )
    viajes_candidatos = viajes.copy()
    viajes_candidatos["direction_key"] = (
        pd.to_numeric(viajes_candidatos["direction_id"], errors="coerce")
        .astype("Int64").astype("string").fillna("sin_sentido")
    )
    columnas_resumen = [
        "route_id", "direction_key", "route_short_name", "route_long_name",
        "numero_viajes", "duracion_p10_min", "duracion_mediana_min",
        "duracion_p90_min", "constante_p50_min", "factor_p50",
        "constante_p80_min", "factor_p80",
        "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
        "observaciones_google", "fuente_modelo",
    ]
    candidatos = viajes_candidatos.merge(
        calculados[columnas_resumen],
        on=["route_id", "direction_key"], how="inner",
        validate="many_to_one", suffixes=("", "_resumen"),
    )
    candidatos["distancia_mediana_min"] = (
        candidatos["duracion_programada_min"]
        - candidatos["duracion_mediana_min"]
    ).abs()
    return (
        candidatos.sort_values(
            ["route_id", "direction_key", "distancia_mediana_min", "trip_id"]
        )
        .drop_duplicates(["route_id", "direction_key"], keep="first")
        .reset_index(drop=True)
    )


def trazar_secuencia_paradas(grupo, trazos_viales):
    """Une las paradas de un viaje por calles; devuelve (coordenadas, km)."""
    ids = grupo["stop_id"].astype(str).tolist()
    puntos = grupo[["stop_lat", "stop_lon"]].astype(float).to_numpy().tolist()
    coordenadas, metros = [puntos[0]], 0.0
    for stop_a, stop_b, punto_a, punto_b in zip(ids, ids[1:], puntos, puntos[1:]):
        trazo = trazos_viales.get((stop_a, stop_b))
        if trazo is None:
            coordenadas.append(punto_b)
            metros += float(haversine_m(*punto_a, *punto_b))
        else:
            coordenadas.extend(trazo["coordenadas"][1:])
            metros += trazo["distancia_red_m"]
    return coordenadas, metros / 1000


def construir_trazos_representativos(
    stop_times, stops, representativos, trazos_viales=None
):
    """Construye los trazos por calles de los viajes representativos."""
    trazos_viales = trazos_viales or {}
    seleccion = representativos[
        [
            "trip_id", "route_id", "direction_key",
            "route_short_name_resumen", "route_long_name_resumen",
            "numero_viajes", "duracion_p10_min",
            "duracion_mediana_min", "duracion_p90_min",
            "constante_p50_min", "factor_p50",
            "constante_p80_min", "factor_p80",
            "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
            "observaciones_google", "fuente_modelo",
        ]
    ].copy()
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(
        puntos["stop_sequence"], errors="raise"
    )
    puntos = (
        puntos.merge(
            stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
            on="stop_id", how="left", validate="many_to_one",
        )
        .merge(seleccion, on="trip_id", how="left", validate="many_to_one")
        .dropna(subset=["stop_lat", "stop_lon"])
        .sort_values(["trip_id", "stop_sequence"])
    )
    trazos = []
    for trip_id, grupo in puntos.groupby("trip_id", sort=False):
        primera = grupo.iloc[0]
        if len(grupo) < 2:
            continue
        coordenadas, distancia_km = trazar_secuencia_paradas(grupo, trazos_viales)
        trazos.append(
            {
                "trip_id": str(trip_id),
                "route_id": str(primera["route_id"]),
                "direction_key": str(primera["direction_key"]),
                "route_short_name": str(primera["route_short_name_resumen"]),
                "route_long_name": str(primera["route_long_name_resumen"]),
                "numero_viajes": int(primera["numero_viajes"]),
                "p10_min": float(primera["duracion_p10_min"]),
                "p50_min": float(primera["duracion_mediana_min"]),
                "p90_min": float(primera["duracion_p90_min"]),
                "estimado_p50_min": float(primera["tiempo_pasajero_p50_min"]),
                "estimado_p80_min": float(primera["tiempo_pasajero_p80_min"]),
                "observaciones_google": int(primera["observaciones_google"]),
                "fuente_modelo": str(primera["fuente_modelo"]),
                "distancia_km": distancia_km,
                "coordenadas": coordenadas,
            }
        )
    return trazos


In [5]:
def construir_tiempos_acumulados(stop_times, stops, representativos):
    """Propaga segmentos corregidos hacia atrás desde el final de cada viaje."""
    columnas_modelo = [
        "trip_id", "route_id", "direction_key",
        "route_short_name_resumen", "constante_p50_min", "factor_p50",
        "constante_p80_min", "factor_p80", "observaciones_google",
        "fuente_modelo",
    ]
    seleccion = representativos[columnas_modelo].copy()
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(puntos["stop_sequence"], errors="raise")
    puntos["arrival_sec"] = gtfs_time_to_seconds(puntos["arrival_time"])
    puntos["departure_sec"] = gtfs_time_to_seconds(puntos["departure_time"])
    puntos = (
        puntos.merge(seleccion, on="trip_id", how="inner", validate="many_to_one")
        .merge(stops[["stop_id", "stop_name"]], on="stop_id", how="left", validate="many_to_one")
        .sort_values(["trip_id", "stop_sequence"])
    )
    resultados = []
    for _, grupo in puntos.groupby("trip_id", sort=False):
        grupo = grupo.copy().reset_index(drop=True)
        siguiente_llegada = grupo["arrival_sec"].shift(-1)
        segmento_gtfs = ((siguiente_llegada - grupo["departure_sec"]) / 60).fillna(0)
        if not segmento_gtfs.iloc[:-1].between(0, 180, inclusive="both").all():
            raise ValueError(f"Segmentos inválidos en el viaje {grupo.iloc[0]['trip_id']}")
        factor_p50 = float(grupo.iloc[0]["factor_p50"])
        factor_p80 = float(grupo.iloc[0]["factor_p80"])
        segmento_p50 = segmento_gtfs * factor_p50
        segmento_p80 = segmento_gtfs * factor_p80
        restante_gtfs = segmento_gtfs.iloc[::-1].cumsum().iloc[::-1]
        restante_p50 = segmento_p50.iloc[::-1].cumsum().iloc[::-1]
        restante_p80 = segmento_p80.iloc[::-1].cumsum().iloc[::-1]
        hay_recorrido = segmento_gtfs.index < len(grupo) - 1
        grupo["next_stop_id"] = grupo["stop_id"].shift(-1)
        grupo["tiempo_segmento_gtfs_min"] = segmento_gtfs
        grupo["tiempo_segmento_p50_min"] = segmento_p50
        grupo["tiempo_segmento_p80_min"] = segmento_p80
        grupo["tiempo_restante_gtfs_min"] = restante_gtfs
        grupo["tiempo_restante_operativo_p50_min"] = restante_p50
        grupo["tiempo_restante_operativo_p80_min"] = restante_p80
        grupo["tiempo_restante_pasajero_p50_min"] = restante_p50 + np.where(
            hay_recorrido, float(grupo.iloc[0]["constante_p50_min"]), 0.0
        )
        grupo["tiempo_restante_pasajero_p80_min"] = restante_p80 + np.where(
            hay_recorrido, float(grupo.iloc[0]["constante_p80_min"]), 0.0
        )
        resultados.append(grupo)
    columnas = [
        "route_id", "route_short_name_resumen", "direction_key", "trip_id",
        "stop_sequence", "stop_id", "stop_name", "next_stop_id",
        "tiempo_segmento_gtfs_min", "tiempo_segmento_p50_min",
        "tiempo_segmento_p80_min", "tiempo_restante_gtfs_min",
        "tiempo_restante_operativo_p50_min",
        "tiempo_restante_operativo_p80_min",
        "tiempo_restante_pasajero_p50_min",
        "tiempo_restante_pasajero_p80_min",
        "constante_p50_min", "factor_p50", "constante_p80_min",
        "factor_p80", "observaciones_google", "fuente_modelo",
    ]
    return pd.concat(resultados, ignore_index=True)[columnas]


### Conexiones entre rutas

Una **conexión en la misma parada** existe cuando dos o más rutas distintas pasan por el mismo `stop_id`; es el único transbordo que usa el Dijkstra de `01` y `02`. Una **conexión a pie** es otra parada a `RADIO_CONEXION_A_PIE_M` metros o menos en línea recta que ofrece alguna ruta que no pasa por la parada de origen. Esta segunda es una referencia para el usuario y no modifica ningún tiempo calculado.

Se usan las paradas de los viajes representativos, es decir, las mismas que forman las líneas del mapa.

In [6]:
def calcular_conexiones(stop_times, stops, representativos, radio_caminata_m):
    """Identifica dónde se cruzan las rutas dibujadas y a qué paradas conectan.

    - Misma parada: dos o más rutas distintas pasan por el mismo stop_id. Es el
      único transbordo que considera el Dijkstra de 01 y 02.
    - A pie: otra parada a no más de `radio_caminata_m` en línea recta que
      ofrece al menos una ruta que no pasa por la parada de origen. Es una
      referencia para el usuario; el cálculo de tiempos no la utiliza.

    Devuelve (servicio, conexiones, paradas_por_trip):
    servicio tiene una fila por parada y ruta-sentido; conexiones, una fila por
    parada de origen y parada conectada; paradas_por_trip, la secuencia de
    paradas de cada viaje representativo.
    """
    seleccion = representativos[
        ["trip_id", "route_id", "direction_key", "route_short_name_resumen"]
    ].rename(columns={"route_short_name_resumen": "route_short_name"})
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(puntos["stop_sequence"], errors="raise")
    puntos = puntos.sort_values(["trip_id", "stop_sequence"])
    paradas_por_trip = {
        str(trip_id): grupo["stop_id"].astype(str).tolist()
        for trip_id, grupo in puntos.groupby("trip_id", sort=False)
    }
    servicio = (
        puntos.merge(seleccion, on="trip_id", how="inner", validate="many_to_one")
        .merge(
            stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
            on="stop_id", how="inner", validate="many_to_one",
        )
        .drop_duplicates(["stop_id", "route_id", "direction_key"])
        [["stop_id", "stop_name", "stop_lat", "stop_lon",
          "route_id", "route_short_name", "direction_key"]]
        .reset_index(drop=True)
    )
    rutas_por_parada = servicio.groupby("stop_id")["route_id"].agg(
        lambda rutas: frozenset(rutas.astype(str))
    )
    nombres_ruta = (
        servicio.drop_duplicates("route_id")
        .set_index("route_id")["route_short_name"].astype(str).to_dict()
    )
    paradas = (
        servicio.drop_duplicates("stop_id")
        .set_index("stop_id")[["stop_name", "stop_lat", "stop_lon"]]
    )

    def texto_rutas(route_ids):
        return ", ".join(sorted(nombres_ruta.get(r, r) for r in route_ids))

    registros = []
    for stop_id, rutas in rutas_por_parada.items():
        if len(rutas) >= 2:
            registros.append({
                "stop_id": stop_id,
                "tipo_conexion": "Misma parada",
                "stop_id_conexion": stop_id,
                "distancia_m": 0.0,
                "route_ids_conexion": sorted(rutas),
            })

    # Distancias en bloques para no crear de golpe la matriz completa.
    ids = paradas.index.to_numpy()
    lat = paradas["stop_lat"].to_numpy(dtype=float)
    lon = paradas["stop_lon"].to_numpy(dtype=float)
    for inicio in range(0, len(ids), 400):
        fin = min(inicio + 400, len(ids))
        distancias = haversine_m(
            lat[inicio:fin, None], lon[inicio:fin, None], lat[None, :], lon[None, :]
        )
        filas, columnas = np.nonzero(distancias <= radio_caminata_m)
        for fila, columna in zip(filas, columnas):
            origen, destino = ids[inicio + fila], ids[columna]
            if origen == destino:
                continue
            rutas_nuevas = rutas_por_parada[destino] - rutas_por_parada[origen]
            if rutas_nuevas:
                registros.append({
                    "stop_id": origen,
                    "tipo_conexion": "A pie",
                    "stop_id_conexion": destino,
                    "distancia_m": float(distancias[fila, columna]),
                    "route_ids_conexion": sorted(rutas_nuevas),
                })

    conexiones = pd.DataFrame(registros, columns=[
        "stop_id", "tipo_conexion", "stop_id_conexion",
        "distancia_m", "route_ids_conexion",
    ])
    conexiones = conexiones.merge(
        paradas, left_on="stop_id", right_index=True, how="left"
    ).merge(
        paradas[["stop_name"]].rename(columns={"stop_name": "stop_name_conexion"}),
        left_on="stop_id_conexion", right_index=True, how="left",
    )
    conexiones["rutas_en_parada"] = conexiones["stop_id"].map(
        lambda s: texto_rutas(rutas_por_parada[s])
    )
    conexiones["rutas_conexion"] = conexiones["route_ids_conexion"].map(texto_rutas)
    # Minutos por calles: línea recta × factor de desvío, a la velocidad de caminata.
    conexiones["minutos_caminando"] = (
        conexiones["distancia_m"] * FACTOR_DESVIO_CAMINATA / VELOCIDAD_CAMINATA_M_MIN
    )
    conexiones = conexiones.sort_values(
        ["stop_id", "tipo_conexion", "distancia_m"], ascending=[True, False, True]
    ).reset_index(drop=True)
    return servicio, conexiones, paradas_por_trip


In [7]:
import json

from branca.element import MacroElement
from jinja2 import Template


class ScriptMapa(MacroElement):
    """Inserta JavaScript propio después de las capas y controles del mapa."""

    _template = Template("{% macro script(this, kwargs) %}{{ this.codigo }}{% endmacro %}")

    def __init__(self, codigo):
        super().__init__()
        self._name = "ScriptMapa"
        self.codigo = codigo


def json_para_script(datos):
    """Serializa datos para incrustarlos de forma segura dentro de <script>."""
    return json.dumps(datos, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")


def colores_por_ruta(route_ids):
    """Asigna a cada route_id un tono distinto con la razón áurea."""
    colores = {}
    for indice, route_id in enumerate(sorted({str(r) for r in route_ids})):
        tono = (indice * 0.618033988749895) % 1.0
        rojo, verde, azul = colorsys.hsv_to_rgb(tono, 0.72, 0.82)
        colores[route_id] = (
            f"#{round(rojo * 255):02x}{round(verde * 255):02x}{round(azul * 255):02x}"
        )
    return colores


# Los marcadores y popups de conexiones se generan en el navegador a partir de
# un solo bloque de datos; así el HTML no repite el mismo contenido miles de veces.
JS_CONEXIONES = """
(function () {
  const D = __DATOS__;
  const capaMisma = __CAPA_MISMA__;
  const capaPie = __CAPA_PIE__;
  const lineas = __LINEAS__;
  const esc = (t) => String(t).replace(/[&<>"']/g, (c) => ({
    "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"
  })[c]);
  const chip = (r, sentido) => {
    const info = D.rutas[r] || [r, "#6b7280"];
    const s = sentido == null ? "" : " · S" + esc(sentido);
    return "<span style='display:inline-block;margin:1px 2px;padding:0 4px;" +
      "border-left:8px solid " + info[1] + ";background:#f3f4f6;border-radius:2px;" +
      "white-space:nowrap'>" + esc(info[0]) + s + "</span>";
  };
  const rutasDe = (s) => new Set(D.paradas[s][3].map((x) => x[0]));
  const caja = (contenido) => "<div style='font-size:12px;max-height:280px;" +
    "overflow-y:auto;min-width:240px'>" + contenido + "</div>";

  function popupParada(s) {
    const p = D.paradas[s];
    const rutas = p[3].slice().sort((a, b) =>
      (D.rutas[a[0]] || [a[0]])[0].localeCompare((D.rutas[b[0]] || [b[0]])[0]) ||
      String(a[1]).localeCompare(String(b[1])));
    let h = "<b>" + esc(p[0]) + "</b> <span style='color:#6b7280'>(parada " +
      esc(s) + ")</span><div style='margin-top:4px'><b>Rutas en esta parada (" +
      rutasDe(s).size + "):</b><br>" + rutas.map((x) => chip(x[0], x[1])).join("") + "</div>";
    const pie = D.apie[s] || [];
    if (pie.length) {
      const max = 12;
      const filas = pie.slice(0, max).map((c) => "<li style='margin:3px 0'>" +
        esc(D.paradas[c[0]][0]) + " <span style='color:#6b7280'>(" + esc(c[0]) + ", " +
        c[1] + " m · ~" + Math.max(1, Math.round(c[1] * D.desvio / D.velocidad)) + " min)</span><br>" +
        c[2].map((r) => chip(r)).join("") + "</li>");
      if (pie.length > max) {
        filas.push("<li style='color:#6b7280'>y " + (pie.length - max) + " paradas más</li>");
      }
      h += "<div style='margin-top:6px'><b>Conexiones a pie (≤ " + D.radio +
        " m en línea recta):</b><ul style='margin:2px 0;padding-left:16px'>" +
        filas.join("") + "</ul></div>";
    }
    return caja(h);
  }

  function popupRuta(trip, propia, encabezado) {
    const filas = [];
    (D.trips[trip] || []).forEach((s) => {
      const otras = [...rutasDe(s)].filter((r) => r !== propia);
      const vistas = new Set([propia, ...otras]);
      const pie = [];
      (D.apie[s] || []).forEach((c) => c[2].forEach((r) => {
        if (!vistas.has(r)) { vistas.add(r); pie.push(r); }
      }));
      if (!otras.length && !pie.length) return;
      let detalle = otras.map((r) => chip(r)).join("");
      if (pie.length) {
        detalle += "<span style='color:#6b7280'> a pie: </span>" + pie.map((r) => chip(r)).join("");
      }
      filas.push("<li style='margin:3px 0'>" + esc(D.paradas[s][0]) + "<br>" + detalle + "</li>");
    });
    const lista = filas.length
      ? "<ul style='margin:2px 0;padding-left:16px'>" + filas.join("") + "</ul>"
      : "<div style='color:#6b7280'>Sin conexiones con otras rutas.</div>";
    return caja(encabezado + "<div style='margin-top:6px'><b>Paradas con conexión (" +
      filas.length + "):</b>" + lista + "</div>");
  }

  // El tamaño crece con el número de rutas y se ajusta al zoom para no tapar las líneas.
  const escala = (z) => (z <= 11 ? 0.45 : z <= 12 ? 0.65 : z <= 13 ? 0.85 : 1.1);
  const marcadores = [];
  D.misma.forEach((s) => {
    const p = D.paradas[s];
    const rutas = rutasDe(s);
    const nombres = [...rutas].map((r) => (D.rutas[r] || [r])[0]).sort().join(", ");
    const base = 3 + Math.min(rutas.size, 12) * 0.45;
    const m = L.circleMarker([p[1], p[2]], {
      radius: base, color: "#111827", weight: 1.5,
      fill: true, fillColor: "#ffffff", fillOpacity: 0.95
    }).bindTooltip(esc(p[0]) + " · " + rutas.size + " rutas: " + esc(nombres))
      .bindPopup(() => popupParada(s), { maxWidth: 360 })
      .addTo(capaMisma);
    marcadores.push([m, base]);
  });
  D.pares.forEach((par) => {
    const a = D.paradas[par[0]], b = D.paradas[par[1]];
    L.polyline([[a[1], a[2]], [b[1], b[2]]], {
      color: "#374151", weight: 2, opacity: 0.8, dashArray: "4 4"
    }).bindTooltip(esc(a[0]) + " ↔ " + esc(b[0]) + " · " + par[2] + " m (~" +
      Math.max(1, Math.round(par[2] * D.desvio / D.velocidad)) + " min)")
      .addTo(capaPie);
  });
  D.solo_pie.forEach((s) => {
    const p = D.paradas[s];
    const m = L.circleMarker([p[1], p[2]], {
      radius: 3, color: "#6b7280", weight: 1, fill: true,
      fillColor: "#ffffff", fillOpacity: 0.95
    }).bindTooltip(esc(p[0]) + " · conexión a pie")
      .bindPopup(() => popupParada(s), { maxWidth: 360 })
      .addTo(capaPie);
    marcadores.push([m, 3]);
  });
  const mapa = __MAPA__;
  const ajustar = () => {
    const f = escala(mapa.getZoom());
    marcadores.forEach(([m, base]) => m.setRadius(base * f));
  };
  mapa.on("zoomend", ajustar);
  ajustar();
  lineas.forEach((l) => l[0].bindPopup(() => popupRuta(l[1], l[2], l[3]), { maxWidth: 360 }));
  // Planificador origen-destino con tiempos de ruta GTFS distribuidos por distancia.
  const estacionTren = L.latLng(20.600611, -100.402184);
  const marcadorEstacion = L.circleMarker(estacionTren, {
    radius: 8, color: "#7c2d12", weight: 2, fillColor: "#fb923c", fillOpacity: 1
  }).bindTooltip("Estación Corregidora (destino inicial)").addTo(mapa);
  let puntoOrigen = null;
  let puntoDestino = estacionTren;
  let paradaOrigenFijada = null;
  let paradaDestinoFijada = null;
  let modoSeleccion = null;
  let marcadorOrigen = null;
  let marcadorDestino = marcadorEstacion;
  let lineaResultado = null;
  const panel = document.getElementById("planificador-ruta");
  const origenLabel = document.getElementById("origen-seleccionado");
  const destinoLabel = document.getElementById("destino-seleccionado");
  const resultado = document.getElementById("resultado-ruta");
  const penalizacionInput = document.getElementById("penalizacion-transbordo");
  const buscadorOrigen = document.getElementById("origen-parada");
  const buscadorDestino = document.getElementById("destino-parada");
  const listaParadas = document.getElementById("paradas-qrobus");
  const idPorOpcion = new Map();
  Object.entries(D.paradas).forEach(([id, parada]) => {
    const opcion = parada[0] + " [" + id + "]";
    idPorOpcion.set(opcion, id);
    const elemento = document.createElement("option");
    elemento.value = opcion;
    listaParadas.appendChild(elemento);
  });
  L.DomEvent.disableClickPropagation(panel);
  L.DomEvent.disableScrollPropagation(panel);

  const distanciaM = (a, b) => mapa.distance(a, b);
  const costoCaminata = (metros) => metros * D.desvio / D.velocidad;
  const paradasCercanas = (punto) => Object.entries(D.paradas).map(([id, p]) => ({
    id, metros: distanciaM(punto, L.latLng(p[1], p[2]))
  })).filter((x) => x.metros <= 800);

  // lineas enlaza cada trip GTFS con su P50 total de la ruta y el sentido.
  const p50PorTrip = {};
  lineas.forEach((l) => {
    const coincidencia = l[3].match(/GTFS P50 ([\\d.]+) min/);
    if (coincidencia) p50PorTrip[String(l[1])] = Number(coincidencia[1]);
  });
  const rutaPorTrip = {};
  lineas.forEach((l) => { rutaPorTrip[String(l[1])] = String(l[2]); });
  const grafo = {};
  Object.entries(D.trips).forEach(([trip, secuencia]) => {
    const totalMin = p50PorTrip[String(trip)];
    if (!totalMin || secuencia.length < 2) return;
    let distanciaTotal = 0;
    for (let i = 0; i < secuencia.length - 1; i++) {
      const a = D.paradas[secuencia[i]], b = D.paradas[secuencia[i + 1]];
      distanciaTotal += distanciaM(L.latLng(a[1], a[2]), L.latLng(b[1], b[2]));
    }
    if (!distanciaTotal) return;
    const ruta = rutaPorTrip[String(trip)];
    for (let i = 0; i < secuencia.length - 1; i++) {
      const desde = secuencia[i], hasta = secuencia[i + 1];
      const a = D.paradas[desde], b = D.paradas[hasta];
      const minutos = totalMin * distanciaM(L.latLng(a[1], a[2]), L.latLng(b[1], b[2])) / distanciaTotal;
      (grafo[desde] ||= []).push({ hasta, ruta, minutos });
    }
  });

  // Cola de prioridad mínima para Dijkstra.
  const crearCola = () => {
    const datos = [];
    return {
      push(item) {
        datos.push(item); let i = datos.length - 1;
        while (i > 0) { const padre = (i - 1) >> 1; if (datos[padre].costo <= item.costo) break; datos[i] = datos[padre]; i = padre; }
        datos[i] = item;
      },
      pop() {
        if (!datos.length) return null;
        const primero = datos[0], ultimo = datos.pop();
        if (datos.length) { let i = 0; while (true) { let h = i * 2 + 1; if (h >= datos.length) break; if (h + 1 < datos.length && datos[h + 1].costo < datos[h].costo) h++; if (datos[h].costo >= ultimo.costo) break; datos[i] = datos[h]; i = h; } datos[i] = ultimo; }
        return primero;
      },
      get length() { return datos.length; }
    };
  };
  const calcularViaje = () => {
    if (!puntoOrigen || !puntoDestino) return;
    const origenes = paradaOrigenFijada
      ? [{ id: paradaOrigenFijada, metros: 0 }]
      : paradasCercanas(puntoOrigen);
    const destinos = paradaDestinoFijada
      ? [{ id: paradaDestinoFijada, metros: 0 }]
      : paradasCercanas(puntoDestino);
    if (!origenes.length || !destinos.length) {
      resultado.innerHTML = "No hay paradas GTFS a 800 m de uno de los puntos seleccionados.";
      if (lineaResultado) mapa.removeLayer(lineaResultado);
      return;
    }
    const aDestino = new Map(destinos.map((x) => [x.id, x]));
    const penalizacion = Math.max(0, Number(penalizacionInput.value) || 0);
    const distancias = new Map(), anteriores = new Map(), cola = crearCola();
    origenes.forEach((x) => {
      const clave = x.id + "|";
      const costo = costoCaminata(x.metros);
      if (!distancias.has(clave) || costo < distancias.get(clave)) {
        distancias.set(clave, costo); anteriores.set(clave, null); cola.push({ clave, costo });
      }
    });
    let mejor = null;
    while (cola.length) {
      const actual = cola.pop();
      if (actual.costo !== distancias.get(actual.clave)) continue;
      const [parada, rutaActual] = actual.clave.split("|");
      const acceso = aDestino.get(parada);
      if (acceso) {
        const total = actual.costo + costoCaminata(acceso.metros);
        if (!mejor || total < mejor.total) mejor = { clave: actual.clave, total, acceso };
      }
      for (const tramo of grafo[parada] || []) {
        const extra = rutaActual && rutaActual !== tramo.ruta ? penalizacion : 0;
        const costo = actual.costo + tramo.minutos + extra;
        const clave = tramo.hasta + "|" + tramo.ruta;
        if (costo < (distancias.get(clave) ?? Infinity)) {
          distancias.set(clave, costo);
          anteriores.set(clave, { anterior: actual.clave, tramo, penalizacion: extra });
          cola.push({ clave, costo });
        }
      }
    }
    if (!mejor) {
      resultado.innerHTML = "No se encontró un recorrido GTFS dirigido entre estos puntos.";
      if (lineaResultado) mapa.removeLayer(lineaResultado);
      return;
    }
    const pasos = [];
    let cursor = mejor.clave;
    while (anteriores.get(cursor)) { const paso = anteriores.get(cursor); pasos.push(paso); cursor = paso.anterior; }
    pasos.reverse();
    const ids = [cursor.split("|")[0], ...pasos.map((p) => p.tramo.hasta)];
    const minutosBus = pasos.reduce((s, p) => s + p.tramo.minutos, 0);
    const cambios = pasos.filter((p) => p.penalizacion > 0).length;
    const metrosOrigen = origenes.find((x) => x.id === ids[0])?.metros || 0;
    const metrosDestino = mejor.acceso.metros;
    const minPie = costoCaminata(metrosOrigen) + costoCaminata(metrosDestino);
    const nombresRutas = [...new Set(pasos.map((p) => (D.rutas[p.tramo.ruta] || [p.tramo.ruta])[0]))];
    const coordenadas = [puntoOrigen, ...ids.map((id) => L.latLng(D.paradas[id][1], D.paradas[id][2])), puntoDestino];
    if (lineaResultado) mapa.removeLayer(lineaResultado);
    lineaResultado = L.polyline(coordenadas, { color: "#111827", weight: 6, opacity: 0.85, dashArray: "8 5" }).addTo(mapa);
    resultado.innerHTML = "<strong>Total: " + mejor.total.toFixed(1) + " min</strong>" +
      "<br>Autobús " + minutosBus.toFixed(1) + " · caminata " + minPie.toFixed(1) + " min" +
      "<br>Transbordos " + cambios + " × " + penalizacion + " min" +
      "<br>Rutas: " + esc(nombresRutas.join(" → "));
    mapa.fitBounds(lineaResultado.getBounds(), { padding: [45, 45], maxZoom: 14 });
  };
  const mostrarPunto = (punto, texto, destino, stopId = null) => {
    if (destino) paradaDestinoFijada = stopId;
    else paradaOrigenFijada = stopId;
    const cercanas = stopId
      ? [{ id: stopId, metros: 0 }]
      : paradasCercanas(punto);
    const cercana = cercanas.sort((a, b) => a.metros - b.metros)[0];
    texto.textContent = cercana
      ? D.paradas[cercana.id][0] + " (" + Math.round(cercana.metros) + " m)"
      : punto.lat.toFixed(5) + ", " + punto.lng.toFixed(5);
    if (destino) {
      if (marcadorDestino) mapa.removeLayer(marcadorDestino);
      marcadorDestino = L.circleMarker(punto, { radius: 8, color: "#7c2d12", weight: 2, fillColor: "#fb923c", fillOpacity: 1 }).addTo(mapa);
      puntoDestino = punto;
    } else {
      if (marcadorOrigen) mapa.removeLayer(marcadorOrigen);
      marcadorOrigen = L.circleMarker(punto, { radius: 8, color: "#064e3b", weight: 2, fillColor: "#34d399", fillOpacity: 1 }).addTo(mapa);
      puntoOrigen = punto;
    }
    calcularViaje();
  };
  marcadores.forEach(([marcador, base, stopId]) => {
    if (stopId == null) return;
    marcador.on("click", (evento) => {
      if (!modoSeleccion) return;
      L.DomEvent.stopPropagation(evento);
      const destino = modoSeleccion === "destino";
      const parada = D.paradas[stopId];
      modoSeleccion = null;
      mostrarPunto(L.latLng(parada[1], parada[2]), destino ? destinoLabel : origenLabel, destino, stopId);
      mapa.closePopup();
    });
  });
  const elegirParadaBuscada = (input, destino) => {
    const id = idPorOpcion.get(input.value);
    if (!id) {
      resultado.textContent = "Elige una parada de las sugerencias para seleccionarla.";
      return;
    }
    const parada = D.paradas[id];
    mostrarPunto(L.latLng(parada[1], parada[2]), destino ? destinoLabel : origenLabel, destino, id);
  };
  buscadorOrigen.addEventListener("change", () => elegirParadaBuscada(buscadorOrigen, false));
  buscadorDestino.addEventListener("change", () => elegirParadaBuscada(buscadorDestino, true));
  document.getElementById("elegir-origen").addEventListener("click", () => {
    modoSeleccion = "origen"; resultado.textContent = "Haz clic en un puntito de parada o en el mapa.";
  });
  document.getElementById("elegir-destino").addEventListener("click", () => {
    modoSeleccion = "destino"; resultado.textContent = "Haz clic en un puntito de parada o en el mapa.";
  });
  penalizacionInput.addEventListener("change", calcularViaje);
  mapa.on("click", (evento) => {
    if (!modoSeleccion) return;
    const destino = modoSeleccion === "destino";
    mostrarPunto(evento.latlng, destino ? destinoLabel : origenLabel, destino);
    modoSeleccion = null;
  });
  marcadorEstacion.bindPopup("Estación Corregidora · destino predeterminado");
})();
"""


def datos_conexiones_js(servicio, conexiones, paradas_por_trip, trip_ids,
                        nombres_ruta, colores_ruta):
    """Prepara el bloque de datos compacto que usa JS_CONEXIONES."""
    servicio = servicio.assign(
        route_id=servicio["route_id"].astype(str),
        sentido=servicio["direction_key"].astype(str).str.replace("sin_sentido", "?"),
    )
    paradas = {}
    for stop_id, grupo in servicio.groupby("stop_id", sort=False):
        primera = grupo.iloc[0]
        paradas[str(stop_id)] = [
            str(primera["stop_name"]),
            round(float(primera["stop_lat"]), 6),
            round(float(primera["stop_lon"]), 6),
            grupo[["route_id", "sentido"]].values.tolist(),
        ]
    a_pie = conexiones.loc[conexiones["tipo_conexion"].eq("A pie")]
    apie = {
        str(stop_id): [
            [str(fila.stop_id_conexion), round(fila.distancia_m), list(fila.route_ids_conexion)]
            for fila in grupo.itertuples(index=False)
        ]
        for stop_id, grupo in a_pie.groupby("stop_id", sort=False)
    }
    # Cada par se dibuja una sola vez aunque conecte en ambos sentidos.
    pares = {}
    for fila in a_pie.itertuples(index=False):
        clave = tuple(sorted((str(fila.stop_id), str(fila.stop_id_conexion))))
        pares.setdefault(clave, [*clave, round(fila.distancia_m)])
    misma = conexiones.loc[
        conexiones["tipo_conexion"].eq("Misma parada"), "stop_id"
    ].astype(str).tolist()
    return {
        "rutas": {r: [nombres_ruta[r], colores_ruta[r]] for r in nombres_ruta},
        "paradas": paradas,
        "apie": apie,
        "pares": list(pares.values()),
        "misma": misma,
        "solo_pie": sorted(set(apie) - set(misma)),
        "trips": {t: paradas_por_trip.get(t, []) for t in trip_ids},
        "radio": RADIO_CONEXION_A_PIE_M,
        "velocidad": VELOCIDAD_CAMINATA_M_MIN,
        "desvio": FACTOR_DESVIO_CAMINATA,
    }


def crear_mapa_tiempos_totales(trazos, salida_html, servicio=None,
                               conexiones=None, paradas_por_trip=None):
    """Dibuja una línea por ruta-sentido con GTFS, P50/P80 y sus conexiones.

    Con `servicio`, `conexiones` y `paradas_por_trip` (de calcular_conexiones)
    agrega dos capas: paradas donde se cruzan rutas y conexiones a pie. Al dar
    clic en una ruta se listan sus paradas de transbordo.
    """
    if not trazos:
        raise ValueError("No existen trazos calculables para crear el mapa.")
    todas_coords = [coord for trazo in trazos for coord in trazo["coordenadas"]]
    centro = [
        float(np.mean([coord[0] for coord in todas_coords])),
        float(np.mean([coord[1] for coord in todas_coords])),
    ]
    mapa = folium.Map(location=centro, zoom_start=11, tiles="OpenStreetMap")
    capa = folium.FeatureGroup(
        name=f"Rutas con tiempo GTFS y calibrado ({len(trazos)})", show=True
    )
    colores_ruta = colores_por_ruta(trazo["route_id"] for trazo in trazos)
    nombres_ruta = {str(t["route_id"]): str(t["route_short_name"]) for t in trazos}
    hay_conexiones = servicio is not None and conexiones is not None

    elementos_leyenda = []
    lineas_js = []
    for trazo in sorted(
        trazos, key=lambda x: (x["route_short_name"], x["direction_key"])
    ):
        color = colores_ruta[str(trazo["route_id"])]
        sentido = trazo["direction_key"].replace("sin_sentido", "sin dato")
        etiqueta = (
            f"Ruta {trazo['route_short_name']} · sentido {sentido} | "
            f"{trazo['distancia_km']:.1f} km por calles | "
            f"GTFS P50 {trazo['p50_min']:.1f} min | "
            f"estimado P50 {trazo['estimado_p50_min']:.1f} min | "
            f"estimado P80 {trazo['estimado_p80_min']:.1f} min | "
            f"Google n={trazo['observaciones_google']} ({trazo['fuente_modelo']})"
        )
        if hay_conexiones:
            etiqueta += " | clic: ver conexiones"
        linea = folium.PolyLine(
            trazo["coordenadas"], color=color, weight=4, opacity=0.78,
            tooltip=etiqueta,
        )
        linea.add_to(capa)
        encabezado = (
            f"<b>Ruta {html.escape(trazo['route_short_name'])} · sentido "
            f"{html.escape(sentido)}</b><br>"
            f"GTFS P50 {trazo['p50_min']:.1f} min · estimado P50 "
            f"{trazo['estimado_p50_min']:.1f} · P80 {trazo['estimado_p80_min']:.1f} min"
        )
        lineas_js.append(
            (linea.get_name(), trazo["trip_id"], str(trazo["route_id"]), encabezado)
        )
        elementos_leyenda.append(
            "<div style='white-space:nowrap;margin:2px 0'>"
            f"<span style='display:inline-block;width:14px;height:4px;"
            f"background:{color};margin-right:6px;vertical-align:middle'></span>"
            f"{html.escape(str(trazo['route_short_name']))} · S{html.escape(sentido)} "
            f"— GTFS {trazo['p50_min']:.1f} · "
            f"P50 {trazo['estimado_p50_min']:.1f} · "
            f"P80 {trazo['estimado_p80_min']:.1f} min</div>"
        )
    capa.add_to(mapa)

    if hay_conexiones:
        datos = datos_conexiones_js(
            servicio, conexiones, paradas_por_trip or {},
            [t["trip_id"] for t in trazos], nombres_ruta, colores_ruta,
        )
        capa_misma = folium.FeatureGroup(
            name=f"Paradas de transbordo: misma parada ({len(datos['misma'])})", show=True
        ).add_to(mapa)
        capa_pie = folium.FeatureGroup(
            name=f"Conexiones a pie ≤ {RADIO_CONEXION_A_PIE_M:g} m ({len(datos['pares'])} pares)",
            show=False,
        ).add_to(mapa)

    folium.LayerControl(collapsed=False).add_to(mapa)
    if hay_conexiones:
        lineas_literal = "[" + ",".join(
            f"[{nombre},{json_para_script(trip)},{json_para_script(ruta)},"
            f"{json_para_script(encabezado)}]"
            for nombre, trip, ruta, encabezado in lineas_js
        ) + "]"
        codigo = (
            JS_CONEXIONES.replace("__DATOS__", json_para_script(datos))
            .replace("__CAPA_MISMA__", capa_misma.get_name())
            .replace("__CAPA_PIE__", capa_pie.get_name())
            .replace("__LINEAS__", lineas_literal)
            .replace("__MAPA__", mapa.get_name())
        )
        mapa.add_child(ScriptMapa(codigo))
    mapa.fit_bounds(todas_coords)
    nota_conexiones = (
        "<div style='margin:4px 0;color:#374151'>"
        "<span style='display:inline-block;width:9px;height:9px;border-radius:50%;"
        "background:#fff;border:2px solid #111827;margin-right:5px;vertical-align:middle'></span>"
        "Transbordo en la misma parada · clic en una ruta o parada para ver "
        "a dónde conecta</div>"
        if hay_conexiones else ""
    )
    leyenda = folium.Element(
        "<div style='position:fixed;bottom:24px;left:24px;z-index:9999;"
        "background:white;border:1px solid #777;border-radius:5px;padding:8px;"
        "font-size:12px;max-height:300px;overflow-y:auto;box-shadow:0 1px 5px #999'>"
        "<b>Tiempo total GTFS y calibrado por ruta y sentido</b>"
        + nota_conexiones
        + "".join(elementos_leyenda)
        + "</div>"
    )
    mapa.get_root().html.add_child(leyenda)
    if hay_conexiones:
        planificador = folium.Element(
            """
            <div id="planificador-ruta" style="position:fixed;top:12px;left:12px;z-index:10000;width:270px;max-width:calc(100vw - 24px);max-height:52vh;overflow-y:auto;background:#fff;border:1px solid #9ca3af;border-radius:8px;padding:9px;box-shadow:0 2px 10px #0003;font:12px/1.3 Arial,sans-serif;color:#111827">
              <strong style="font-size:14px">Calcular viaje</strong>
              <div style="margin:8px 0 5px">Origen: <span id="origen-seleccionado">sin seleccionar</span></div>
              <button id="elegir-origen" type="button" style="padding:5px 8px;cursor:pointer">Elegir origen</button>
              <label style="display:block;margin-top:5px">O busca una parada QroBus
                <input id="origen-parada" list="paradas-qrobus" placeholder="Nombre o ID de parada" style="width:100%;box-sizing:border-box;padding:4px">
              </label>
              <div style="margin:8px 0 5px">Destino: <span id="destino-seleccionado">Estación Corregidora</span></div>
              <button id="elegir-destino" type="button" style="padding:5px 8px;cursor:pointer">Elegir destino</button>
              <label style="display:block;margin-top:5px">O busca una parada QroBus
                <input id="destino-parada" list="paradas-qrobus" placeholder="Nombre o ID de parada" style="width:100%;box-sizing:border-box;padding:4px">
              </label>
              <datalist id="paradas-qrobus"></datalist>
              <label style="display:block;margin-top:9px">Penalización por transbordo (min)
                <input id="penalizacion-transbordo" type="number" min="0" step="1" value="15" style="width:55px;margin-left:5px;padding:3px">
              </label>
              <div id="resultado-ruta" aria-live="polite" style="margin-top:9px;color:#374151">Selecciona un origen para calcular el viaje.</div>
              <div style="margin-top:7px;color:#6b7280;font-size:11px">Tiempos por tramo aproximados. Caminata de acceso: hasta 800 m.</div>
            </div>
            """
        )
        mapa.get_root().html.add_child(planificador)
    salida_html.parent.mkdir(parents=True, exist_ok=True)
    mapa.save(salida_html)
    return mapa


## 1. Carga del GTFS


In [8]:
stop_times = pd.read_csv(
    GTFS_DIR / "stop_times.txt",
    dtype={"trip_id": "string", "stop_id": "string"},
)
trips = pd.read_csv(
    GTFS_DIR / "trips.txt",
    dtype={"trip_id": "string", "route_id": "string"},
)
routes = pd.read_csv(GTFS_DIR / "routes.txt", dtype={"route_id": "string"})
stops = pd.read_csv(GTFS_DIR / "stops.txt", dtype={"stop_id": "string"})
print(f"Stop times: {len(stop_times):,}")
print(f"Viajes catalogados: {trips['trip_id'].nunique():,}")
print(f"Rutas catalogadas: {routes['route_id'].nunique():,}")


Stop times: 1,326,009
Viajes catalogados: 32,536
Rutas catalogadas: 217


## 2. Duración completa de cada viaje

Para cada `trip_id` se resta la salida de la primera parada a la llegada de la última. Las horas GTFS mayores a 24:00 se conservan correctamente.


In [9]:
viajes = calcular_duraciones_viajes(stop_times, trips, routes)
resumen = agregar_rutas_sin_viajes(
    resumir_por_ruta_sentido(viajes), routes
)
resumen.to_csv(RESUMEN_PATH, index=False)
viajes.to_csv(VIAJES_PATH, index=False)
print(f"Viajes analizados: {viajes['trip_id'].nunique():,}")
print(f"Rutas con viajes: {viajes['route_id'].nunique():,}")
print(
    "Rutas del catálogo sin viajes: "
    f"{resumen.loc[resumen['estado_calculo'].ne('calculado'), 'route_id'].nunique():,}"
)
display(
    resumen.loc[resumen["estado_calculo"].eq("calculado")].head(10)
)


Viajes analizados: 32,536
Rutas con viajes: 117
Rutas del catálogo sin viajes: 100


,route_id,route_short_name,route_long_name,direction_id,numero_viajes,numero_servicios,duracion_min_min,duracion_p10_min,duracion_mediana_min,duracion_promedio_min,duracion_p90_min,duracion_max_min,paradas_mediana,estado_calculo
42,386,AIQ01,Alameda - Aeropuerto Internacional De Queretaro,0.0,33,3,55.30,55.30,73.73,67.70,73.73,73.73,21.0,calculado
43,386,AIQ01,Alameda - Aeropuerto Internacional De Queretaro,1.0,33,3,52.63,52.63,70.02,63.70,70.02,70.02,28.0,calculado
44,96,C21,Las Flores - Fracc Fray Junipero Serra,0.0,204,3,93.22,93.22,104.88,102.89,109.30,109.30,69.0,calculado
45,96,C21,Las Flores - Fracc Fray Junipero Serra,1.0,204,3,93.73,93.73,98.28,98.95,103.87,103.87,64.0,calculado
46,97,C22,Fracc Santuarios - Penal,0.0,160,3,76.47,79.98,90.67,89.11,95.12,95.12,67.0,calculado
47,97,C22,Fracc Santuarios - Penal,1.0,160,3,79.73,85.18,94.78,93.83,101.60,101.60,70.0,calculado
48,98,C23,Lomas De La Cruz - Terminal De Autobuses (taq),0.0,188,3,91.78,91.78,98.15,96.74,99.87,99.87,69.0,calculado
49,98,C23,Lomas De La Cruz - Terminal De Autobuses (taq),1.0,188,3,80.47,80.47,90.95,88.05,91.85,91.85,65.0,calculado
50,319,C24,La Pradera - Bosques Del Cimatario,0.0,173,3,65.08,65.08,69.65,69.93,74.20,74.20,58.0,calculado
51,319,C24,La Pradera - Bosques Del Cimatario,1.0,175,3,60.77,60.77,73.45,69.54,73.45,73.45,51.0,calculado


## 3. Aprendizaje de la constante y el factor de atraso

Cada observación histórica compara una ETA de Google con el tiempo GTFS hacia Corregidora. Se ajustan dos modelos: `P50 = constante_P50 + factor_P50 × GTFS` y `P80 = constante_P80 + factor_P80 × GTFS`. La constante se aplica una sola vez por viaje y el factor se aplica a cada segmento.

La atribución por ruta usa `route_id_principal` del camino GTFS; Google no confirma en la matriz que haya utilizado esa misma línea. Por ello la calibración es un **proxy**. Las rutas con muestra insuficiente utilizan el modelo global y las demás se suavizan hacia él para reducir sobreajuste.


In [10]:
if OBSERVACIONES_GOOGLE_PATH.exists():
    observaciones_crudas = pd.read_csv(
        OBSERVACIONES_GOOGLE_PATH, dtype={"stop_id": "string"}
    )
    observaciones_modelo = preparar_observaciones_google(observaciones_crudas)
    calibracion = calibrar_modelos_por_ruta(observaciones_modelo)
else:
    observaciones_modelo = pd.DataFrame()
    calibracion = pd.DataFrame([{
        "route_id": "__GLOBAL__", "observaciones_google": 0,
        "rango_gtfs_observado_min": 0.0, "peso_modelo_ruta": 0.0,
        "constante_p50_min": 0.0, "factor_p50": 1.0,
        "constante_p80_min": 0.0, "factor_p80": 1.0,
        "fuente_modelo": "GTFS sin histórico de Google",
    }])

modelo_global = calibracion.loc[calibracion["route_id"].eq("__GLOBAL__")].iloc[0]
modelos_ruta = calibracion.loc[calibracion["route_id"].ne("__GLOBAL__")].copy()
resumen["route_id"] = resumen["route_id"].astype("string")
resumen = resumen.merge(modelos_ruta, on="route_id", how="left", validate="many_to_one")
for columna in ["constante_p50_min", "factor_p50", "constante_p80_min", "factor_p80"]:
    resumen[columna] = resumen[columna].fillna(float(modelo_global[columna]))
resumen["observaciones_google"] = resumen["observaciones_google"].fillna(0).astype(int)
resumen["rango_gtfs_observado_min"] = resumen["rango_gtfs_observado_min"].fillna(0.0)
resumen["peso_modelo_ruta"] = resumen["peso_modelo_ruta"].fillna(0.0)
resumen["fuente_modelo"] = resumen["fuente_modelo"].fillna("global")
resumen["tiempo_operativo_p50_min"] = (
    resumen["duracion_mediana_min"] * resumen["factor_p50"]
)
resumen["tiempo_operativo_p80_min"] = (
    resumen["duracion_mediana_min"] * resumen["factor_p80"]
)
resumen["tiempo_pasajero_p50_min"] = (
    resumen["constante_p50_min"] + resumen["tiempo_operativo_p50_min"]
)
resumen["tiempo_pasajero_p80_min"] = (
    resumen["constante_p80_min"] + resumen["tiempo_operativo_p80_min"]
)
columnas_estimadas = [
    "constante_p50_min", "factor_p50", "constante_p80_min", "factor_p80",
    "tiempo_operativo_p50_min", "tiempo_operativo_p80_min",
    "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
]
resumen[columnas_estimadas] = resumen[columnas_estimadas].round(2)
calibracion.to_csv(CALIBRACION_PATH, index=False)
resumen.to_csv(RESUMEN_PATH, index=False)
print(f"Observaciones válidas usadas: {len(observaciones_modelo):,}")
print(f"Rutas con calibración propia suavizada: {len(modelos_ruta):,}")
print(
    f"Modelo global P50: {modelo_global['constante_p50_min']:.2f} + "
    f"{modelo_global['factor_p50']:.3f} × GTFS"
)
print(
    f"Modelo global P80: {modelo_global['constante_p80_min']:.2f} + "
    f"{modelo_global['factor_p80']:.3f} × GTFS"
)
display(
    resumen.loc[resumen["estado_calculo"].eq("calculado"), [
        "route_short_name", "direction_id", "duracion_mediana_min",
        "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
        "observaciones_google", "fuente_modelo",
    ]].head(10)
)


Observaciones válidas usadas: 9,645
Rutas con calibración propia suavizada: 55
Modelo global P50: 0.00 + 1.000 × GTFS
Modelo global P80: 6.00 + 1.000 × GTFS


,route_short_name,direction_id,duracion_mediana_min,tiempo_pasajero_p50_min,tiempo_pasajero_p80_min,observaciones_google,fuente_modelo
42,AIQ01,0.0,73.73,73.73,79.73,0,global
43,AIQ01,1.0,70.02,70.02,76.02,0,global
44,C21,0.0,104.88,107.34,111.85,148,ruta suavizada con global
45,C21,1.0,98.28,100.74,105.25,148,ruta suavizada con global
46,C22,0.0,90.67,90.67,92.93,166,ruta suavizada con global
47,C22,1.0,94.78,94.78,97.04,166,ruta suavizada con global
48,C23,0.0,98.15,98.15,100.44,162,ruta suavizada con global
49,C23,1.0,90.95,90.95,93.24,162,ruta suavizada con global
50,C24,0.0,69.65,69.65,71.63,203,ruta suavizada con global
51,C24,1.0,73.45,73.45,75.43,203,ruta suavizada con global


## 4. Propagación por segmentos hacia la primera parada

Para cada ruta y sentido se elige el viaje más cercano a la mediana GTFS. Cada segmento se multiplica por su factor P50/P80 y luego se realiza una suma acumulada inversa desde la última parada. La constante se añade una sola vez al tiempo del pasajero; no se repite en cada segmento. Los segmentos corregidos son estimaciones modeladas: la matriz actual de Google entrega tiempos completos origen-destino, no observaciones independientes de cada segmento.


In [11]:
representativos = seleccionar_viajes_representativos(viajes, resumen)
tiempos_acumulados = construir_tiempos_acumulados(
    stop_times, stops, representativos
)
tiempos_acumulados.to_csv(ACUMULADOS_PATH, index=False)
assert tiempos_acumulados["tiempo_restante_operativo_p50_min"].ge(
    tiempos_acumulados["tiempo_restante_gtfs_min"] - 1e-9
).all()
assert tiempos_acumulados["tiempo_restante_pasajero_p80_min"].ge(
    tiempos_acumulados["tiempo_restante_pasajero_p50_min"] - 1e-9
).all()
print(f"Paradas acumuladas exportadas: {len(tiempos_acumulados):,}")
print(f"Archivo: {ACUMULADOS_PATH}")
display(tiempos_acumulados.head(10))


Paradas acumuladas exportadas: 8,071
Archivo: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/tiempos_acumulados_paradas_rutas.csv


,route_id,route_short_name_resumen,direction_key,trip_id,stop_sequence,stop_id,stop_name,next_stop_id,tiempo_segmento_gtfs_min,tiempo_segmento_p50_min,...,tiempo_restante_operativo_p50_min,tiempo_restante_operativo_p80_min,tiempo_restante_pasajero_p50_min,tiempo_restante_pasajero_p80_min,constante_p50_min,factor_p50,constante_p80_min,factor_p80,observaciones_google,fuente_modelo
0,386,AIQ01,1,10,1,2033,Prol. Luis Pasteur/Constituyentes,3573,0.900000,0.900000,...,65.683333,65.683333,65.683333,71.683333,0.0,1.0,6.0,1.0,0,global
1,386,AIQ01,1,10,2,3573,Av. Luis Pateur Sur/José A. Bustamante,2034,0.366667,0.366667,...,64.783333,64.783333,64.783333,70.783333,0.0,1.0,6.0,1.0,0,global
2,386,AIQ01,1,10,3,2034,Prol. Luis Pasteu /Chichén Itzá,2035,0.450000,0.450000,...,64.416667,64.416667,64.416667,70.416667,0.0,1.0,6.0,1.0,0,global
3,386,AIQ01,1,10,4,2035,Prol. Luis Pasteur/José Sotelo,3341,1.416667,1.416667,...,63.966667,63.966667,63.966667,69.966667,0.0,1.0,6.0,1.0,0,global
4,386,AIQ01,1,10,5,3341,Carr. Lateral No. 57/Prol. del Cimatario,2751,0.766667,0.766667,...,62.550000,62.550000,62.550000,68.550000,0.0,1.0,6.0,1.0,0,global
5,386,AIQ01,1,10,6,2751,Lateral Querétaro A Mexico/Comercial Mexicana,3783,7.666667,7.666667,...,61.783333,61.783333,61.783333,67.783333,0.0,1.0,6.0,1.0,0,global
6,386,AIQ01,1,10,7,3783,San Isidro Miranda/Autopista México/Querétaro,3863,2.116667,2.116667,...,54.116667,54.116667,54.116667,60.116667,0.0,1.0,6.0,1.0,0,global
7,386,AIQ01,1,10,8,3863,Carr. Federal No. 57/Calle San Ignacio,5213,1.016667,1.016667,...,52.000000,52.000000,52.000000,58.000000,0.0,1.0,6.0,1.0,0,global
8,386,AIQ01,1,10,9,5213,Av. Estado de México/Av. Aguascalientes,3784,0.550000,0.550000,...,50.983333,50.983333,50.983333,56.983333,0.0,1.0,6.0,1.0,0,global
9,386,AIQ01,1,10,10,3784,Carr. Federal No. 57/Calle Jalisco,3864,1.100000,1.100000,...,50.433333,50.433333,50.433333,56.433333,0.0,1.0,6.0,1.0,0,global


## 5. Mapa por ruta y sentido, con conexiones

Cada ruta utiliza un color diferente. El tooltip y la leyenda comparan la mediana GTFS con los tiempos estimados P50 y P80 del pasajero e indican si el modelo procede de la ruta o del ajuste global.

Además:

- **Clic en una ruta:** lista sus paradas donde se puede transbordar y a qué rutas, en la misma parada o caminando.
- **Capa *Paradas de transbordo: misma parada*:** puntos negros donde se cruzan dos o más rutas; el tamaño crece con el número de rutas. El clic muestra las rutas que pasan y las paradas cercanas a las que se puede caminar.
- **Capa *Conexiones a pie* (apagada al inicio):** líneas punteadas entre paradas cercanas que ofrecen rutas distintas.

La tabla completa se exporta a `data/processed/conexiones_paradas_rutas.csv`.

In [12]:
representativos = seleccionar_viajes_representativos(viajes, resumen)
if TRAZOS_PATH.exists():
    trazos_viales = cargar_trazos(TRAZOS_PATH)
else:
    trazos_viales = {}
    print(
        "Falta segmentos_red_vial.csv: las rutas se dibujan en línea recta. "
        "Ejecute 01_PromedioRutas.ipynb para trazarlas sobre las calles."
    )
trazos = construir_trazos_representativos(
    stop_times, stops, representativos, trazos_viales
)
servicio_paradas, conexiones, paradas_por_trip = calcular_conexiones(
    stop_times, stops, representativos, RADIO_CONEXION_A_PIE_M
)
conexiones.assign(
    route_ids_conexion=conexiones["route_ids_conexion"].map(" ".join)
).to_csv(CONEXIONES_PATH, index=False, encoding="utf-8-sig")
misma_parada = conexiones[conexiones["tipo_conexion"].eq("Misma parada")]
a_pie = conexiones[conexiones["tipo_conexion"].eq("A pie")]
print(f"Paradas donde se cruzan 2 o más rutas: {len(misma_parada):,}")
print(f"Paradas con al menos una conexión a pie: {a_pie['stop_id'].nunique():,}")
print(f"Conexiones exportadas: {CONEXIONES_PATH}")
display(
    misma_parada.assign(numero_rutas=misma_parada["route_ids_conexion"].map(len))
    .sort_values(["numero_rutas", "stop_name"], ascending=[False, True])
    [["stop_id", "stop_name", "numero_rutas", "rutas_en_parada"]]
    .head(10)
)

mapa_tiempos = crear_mapa_tiempos_totales(
    trazos, MAPA_PATH, servicio_paradas, conexiones, paradas_por_trip
)
print(f"Combinaciones ruta-sentido dibujadas: {len(trazos):,}")
print(f"Copia HTML: {MAPA_PATH}")
mapa_tiempos


Paradas donde se cruzan 2 o más rutas: 1,527
Paradas con al menos una conexión a pie: 1,148
Conexiones exportadas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/conexiones_paradas_rutas.csv


,stop_id,stop_name,numero_rutas,rutas_en_parada
340,1921,Avenida Ignacio Zaragoza/Comercial Mexicana,32,"C22, C24, C26, C27, C28, C31, C32, C33, C34, C..."
336,1920,Santa Rosa de Viterbo,29,"C24, C26, C27, C28, C31, C32, C33, C34, C39, C..."
357,1926,Ignacio Zaragoza/Ignacio Allende,28,"C24, C26, C27, C28, C31, C32, C33, C34, C39, C..."
353,1925,Ignacio Zaragoza/Vicente Guerrero Salesiano,28,"C24, C26, C27, C28, C31, C32, C33, C34, C39, C..."
333,1919,Vicente Guerrero,27,"C24, C26, C27, C28, C31, C32, C33, C39, C40, C..."
330,1918,Vergara,23,"C26, C28, C31, C32, C33, C39, C40, C42, C48, C..."
689,2203,Alameda,21,"C26, C28, C30, C31, C33, C34, C35, C39, C42, C..."
687,2201,Ignacio Zaragoza/Imss,21,"C23, C28, C39, C40, C42, C46, C48, C50, C68, L..."
671,2195,Ignacio Zaragoza/Telmex,21,"C21, C22, C23, C28, C39, C40, C42, C46, C48, C..."
349,1924,Ignacio Zaragoza/Ezequiel Montes,19,"C23, C26, C28, C32, C39, C40, C50, C68, L151, ..."


Combinaciones ruta-sentido dibujadas: 218
Copia HTML: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_tiempos_totales_rutas_gtfs.html
